# Legal-BERT on SCOTUS (Colab)
Runs `src/transformer.py` (chunked Legal-BERT) on a Colab GPU.
**Before starting:** Runtime -> Change runtime type -> **T4 GPU** (or better).
Upload `colab_upload.zip` (the project's `src/` folder, made locally) when asked in step 2.

In [ ]:
# 1. GPU check
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# 2. Upload colab_upload.zip and unpack (creates ./src)
from google.colab import files
up = files.upload()            # choose colab_upload.zip
!unzip -o colab_upload.zip
!ls src

In [ ]:
# 3. Dependencies (torch is preinstalled on Colab)
!pip -q install transformers pandas pyarrow scikit-learn matplotlib seaborn tqdm

In [ ]:
# 4. Download SCOTUS parquet (~173 MB) from Hugging Face
!python -m src.data --dataset scotus

In [ ]:
# 5. Smoke test (about 1 minute). Metrics are meaningless; it only checks the pipeline runs.
!python -m src.transformer --dataset scotus --mode finetune --quick

In [ ]:
# 6. Frozen embeddings + LR (also saves embeddings for similar-case retrieval). ~10 min on a T4.
!python -m src.transformer --dataset scotus --mode embed

In [ ]:
# 7. Fine-tune (3 epochs). ~1-2 h on a T4. Keep the tab open / avoid idle disconnects.
# If you hit CUDA out-of-memory, add:  --batch-size 4
!python -m src.transformer --dataset scotus --mode finetune --epochs 3

In [ ]:
# 8. Zip results + saved artifacts and download them
!zip -r colab_outputs.zip results models/scotus_legalbert_embeddings.npz
from google.colab import files
files.download('colab_outputs.zip')
# The fine-tuned weights (~440 MB) are NOT in the zip; to download them too:
# files.download('models/scotus_legalbert_finetuned.pt')